# Data pipeline and Snakemake

Today we discuss how to structure a data analysis pipeline, which is the logic of it, how to manage data and automate the analysis with **workflow management tools**.

In particular, the one we are going to use is called **snakemake**.

This automation can be done at several levels:

* do not manually change the data;
* do not trow away the results of the analysis;
* do not check them just with your eyes;
* manually join the various steps of the analysis.

In an ideal world, all the analysis and preliminary reports should be done with a single push of the button.

## Reproducible data analysis

Anyway, automation is just one of the features characterizing a data analysis pipeline.

In general, the ultimate goal is to generate a **reproducible analysis**, meaning that you have to take into account three aspects:

* **automation** - you have to be able to execute all the steps of the analysis without manual intervention
* **scalability** - you have to able to handle parallelization in order to effectively use the available resources
* **portability** - you have to be able to easily execute analyses on different platforms, systems or infrastructures

## Data pipelines

First of all, we need to discuss on how data can be connected together within a data analysis pipeline.

We can use a simple classification to discuss them. These are not formal, official name, just useful to discuss them and their role, and they reflect the tipical use in a pipeline:

1. metadata
2. raw data
3. source code
4. source data
5. usage data
6. intermediate data
7. temporary data

### 1 - Metadata

Data related to the data that are going to be analyzed.

Typically stored as a simple text file, it usually describes why the data have been collected, the experimental procedure, when, who did it, and so on.

It might seem trivial, but after few years one might find themselves with a disk full of TBs of data that they have no idea what to do with because they don't know what data they are.

Data whose reason for existence is unknown are as functionally valid as data that have been completely deleted.

Often it also include the hash values of raw data and source code to make sure of their integrity.

Associamo ai dati raw un file di testo per indicare la sorgente, la data e tutte le informazioni che permettono di capire cosa c'è nei dati.

### 2 - Raw data

These are the original data that came out of your experiments. Directly from the machine, with **no human intervention, AT ALL**. One will not use these in the analysis, but will **process to make them suitable for analsys**. They might be in the most weird formats, and if they need some specific program to be read, it would be a good idea to keep track of that as well (if possible, store the program as well).

These data ARE SACRED. **They should be preserved and never changed**.

If one happens to get new versions (for example a measure has been repeated and updated), do not overwrite them, but store them alongside the previous version.

### 3 - Source code

It might seems trivial, but the source code to process and analyze the data contains informations about the data that might not be available elsewhere:

* how should the data be read and processed?
* how were the errors corrected?
* what are possible things to be aware of when using them?
* which analysis have already been done?

This are information about the data and should be treated as such, and are often as important as the RAW data.

This is another reason to use the version control!

### 4 - Source data

Once one has the RAW data, they can be processed in a useful format using the script generated by the source code. This format is the one that you will load with other program to preprocess them in the format used for the analysis.

Using this script one can:

* merge files broken down in various way (for example data divided by subjects), adding info about the origin
* correct mistakes in the RAW files (don't edit them!)
* restructure them in a data structure easy to maintain in the long term (my preference goes for text files such as **csv**)

In this phase you should **try to maintain the value of the data as close to the RAW as possible** (but not necessarely their structure), **so avoid any kind of preprocessing, such as detrending, normalizations** and so on.

If the RAW are already in a reasonable format, you could consider them also as your source data, but I never seen it happens. Most of the time, when this happens, those data are not really raw data, but preprocessed ones...try to understand what have been done to them!

### 5 - Usage data

This is the actual starting point of our analysis.

Starting from the source data, we can compose them in a format more comfortable for the analysis that we want, without worrying about database normalization, information duplication and so on.

These data are typically going to be generated only once for each analysis, unless one finds problems with the underlying concept of the analysis (so one needs a different format) or in the assumptions of the data (so one might needs to go and reprocess the RAW data into the source data).

Different analysis might need different usage data.

### 6 - Intermediate data

Your analysis will probably be composed of several steps, such as data split, normalization, detrending and so on.

**After each one of these steps, is it good practice to keep track of the results with an intermediate dataset**, so that you can recover your analysis at any point without having to run everything that led to that result.

Usually the only issue that one might have losing this data is having to re-run the analysis, requiring more time.

### 7 - Temporary data

These are the intermediate results of analysis steps, similarly to the intermediate data, but with the **explicit goal of being deleted after the end of each step**.

These might be generated for example by a parallel distributed algorithm, that analyze one patient at the time and then merge the results at the end of the run. Once the final table has been generated, there is no reason to keep the other files around consuming space, and thus they need to be removed

## Snakemake

Snakemake is a python reimplementation of the ideas behind the classing GNU make, traditionally used for code compilation.

Snakemake allows to **automatize** complicated data processing pipelines in a very comfortable fashion, then to **scale them up** to parallel processing and distributed computing (included grids) with few lines of code.

**It allows to connect simple bash (or powershell) scripts, python and R programs and to mix raw python code in the pipeline code**.

### Other systems for workflow management

Snakemake is only one of several systems for worklow management.

Is the one that in my personal opinion is closer to our needs, but I suggest you to check also the others if you need to.

Sone famous other libraries (not all targeted to python) are:

* Luigi
* Apache Airflow
* BigDataScript
* Dask
* Nextflow

A more comprehensive list can be found at: https://github.com/pditommaso/awesome-pipeline

### Fundamental elements of Snakemake

The fundamental element of snakemake is the **rule**, that represent a program (typically a script), and the program input and output files.

**Each rule get executed as a separated process**, orchestrated by the main snakemake execution.

Each rule can have several subsections, of which the most important are:

* **output**: the list of files that the list will generate (is a promise, you will actually have to create them inside the rule)
* **input**: the list of files required by the rule to be executed to generate the promised output files
* **run** or **shell**: execute one or more shell (or python) commands

### How Snakemake proceeds

By default, if nothing else is specified, snakemake tries to execute the rule all.

**If the output files of a rule exists already, the rule is skipped** (unless you force snakemake to).

If the input required to generate those output do not exists, snakmake tries to find another rule that can have those input file as output of its executions.

This structure is called pull (I specify my arrive point, and try to guess how to get there), compared to the push model, in which one specifies explicitely how the data flows from one step to the other.

Working with a pull approach takes some time to get used to, but can provide several advantages for heavy computations.

In [6]:
%%file Snakefile

rule all:
    shell:
        "echo 'hello world' > result.txt"

Writing Snakefile


In [7]:
!snakemake --cores 1

Assuming unrestricted shared filesystem usage.
host: Filippos-MacBook-Air.local
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 1 (use --cores to define parallelism)
Rules claiming more threads will be scaled down.
Job stats:
job      count
-----  -------
all          1
total        1

Select jobs to execute...
Execute 1 jobs...

[Fri May  8 10:40:58 2026]
localrule all:
    jobid: 0
    reason: Rules with neither input nor output files are always executed.
    resources: tmpdir=/var/folders/zh/qvndpk213k531zrnc3m69jwr0000gn/T
[Fri May  8 10:40:58 2026]
Finished jobid: 0 (Rule: all)
1 of 1 steps (100%) done
Complete log(s): /Users/filippodiludovico/Library/Mobile Documents/com~apple~CloudDocs/Uni/Software and Computing/03.notes/.snakemake/log/2026-05-08T104057.629159.snakemake.log


In [8]:
!ls

01.md
01_random_number_generation_2025_2026.slides.html
02.ipynb
03.ipynb
04.ipynb
04_great_expectations_cleaned.txt
04_markov_chains_for_text_generation_2025_2026.slides.html
05.dna_assembly.slides.html
06.Snakemake_pipeline.html
06.ipynb
MNIST
Snakefile
result.txt


If the output of a rule already exists, and is more recent than the input file, the rule will not be executed.

This property is called **idempotency**, and makes the execution of the script easier to predict.

It is still possible to force snakemake's hand if one needs to (study in the manual how).

In [9]:
%%file Snakefile

rule all:
    output:
        "result.txt"
    shell:
        "echo 'hello world' > {output}"

Overwriting Snakefile


In [11]:
!snakemake --cores 1

Assuming unrestricted shared filesystem usage.
host: Filippos-MacBook-Air.local
Building DAG of jobs...
Nothing to be done (all requested files are present and up to date).


In [12]:
!cat result.txt

hello world


If the required input files:

* do not exists
* there is no rule that can generate them

snakemake will crash with an error.

Ora per creare il file result ??

In [ ]:
%%file Snakefile

rule all:
    input:
        "partial_1.txt",
        "partial_2.txt"
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"

Overwriting Snakefile


In [16]:
!snakemake --cores 1

Assuming unrestricted shared filesystem usage.
host: Filippos-MacBook-Air.local
Building DAG of jobs...
MissingInputException in rule all in file "/Users/filippodiludovico/Library/Mobile Documents/com~apple~CloudDocs/Uni/Software and Computing/03.notes/Snakefile", line 2:
Missing input files for rule all:
    output: result.txt
    affected files:
        partial_2.txt
        partial_1.txt


Let's see how would a script look using two rules, one to generate the partial files and one to process them.

In [7]:
!rm result.txt

rm: result.txt: No such file or directory


Qui ho due rule: all e create_partials, che crea due file output usando pyhton

In [18]:
%%file Snakefile

rule all:
    input:
        "partial_1.txt",
        "partial_2.txt"
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"
        
rule create_partials:
    output:
        "partial_1.txt",
        "partial_2.txt"
    run:
        for filename in output:
            with open(filename, 'w') as file:
                print("the result of {}".format(filename), file=file)

Overwriting Snakefile


In [19]:
!snakemake --cores 1

Assuming unrestricted shared filesystem usage.
host: Filippos-MacBook-Air.local
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 1 (use --cores to define parallelism)
Rules claiming more threads will be scaled down.
Job stats:
job                count
---------------  -------
create_partials        1
all                    1
total                  2

Select jobs to execute...
Execute 1 jobs...

[Fri May  8 10:44:57 2026]
localrule create_partials:
    output: partial_1.txt, partial_2.txt
    jobid: 1
    reason: Missing output files: partial_1.txt, partial_2.txt
    resources: tmpdir=/var/folders/zh/qvndpk213k531zrnc3m69jwr0000gn/T
[Fri May  8 10:44:59 2026]
Finished jobid: 1 (Rule: create_partials)
1 of 2 steps (50%) done
Select jobs to execute...
Execute 1 jobs...

[Fri May  8 10:44:59 2026]
localrule all:
    input: partial_1.txt, partial_2.txt
    output: result.txt
    jobid: 0
    reason: Missing output files: result.txt; Input files updated by another job: partial_

In [ ]:
!ls

In [ ]:
!cat partial_1.txt

In [ ]:
!cat partial_2.txt

In [ ]:
!cat result.txt

If the intermediate files already exists, the rule will not be executed again.

In [ ]:
!rm result.txt

In [ ]:
!snakemake

## Wildcards

In this script the same python function create both files, one at the time, even if there is no need to wait.

They could be generated concurrently, without waiting. To be able to do this I have to specify a rule that generate a generic partial file, using wildcards, and let snakemake do the rest.

Wildcards might cause severe headaches, do not try to get too fancy with them!

In [ ]:
!rm *.txt

In questo caso sto considerando tutti i file che hanno nome "partial_{number}.txt".

In [ ]:
%%file Snakefile

rule all:
    input:
        "partial_1.txt",
        "partial_2.txt",
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"
        
rule create_partials:
    output:
        out="partial_{number}.txt"
    run:
        filename = output.out
        with open(filename, 'w') as file:
            print("the result of {}".format(filename), file=file)

In [ ]:
!snakemake

To be able to use wildcards I need somewhere an **expand** command, that tells snakemake exactly which files to search for.

I can have several wildcards at the same time, the important thing is to give an example that allows to initialize all of them.

There are methods for automatic inference of wildcards, but I suggest to start using the explicit ones.

In [ ]:
%%file Snakefile

numbers = [1, 2, 3, 4]

rule all:
    input:
        expand("partial_{number}.txt", number=numbers)
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"
        
rule create_partials:
    output:
        out = "partial_{number}.txt"
    run:
        filename = output.out
        with open(filename, 'w') as file:
            print("the result of {}".format(filename), file=file)

In [ ]:
!rm result.txt

In [ ]:
!snakemake

## Flow chart of the execution plan

Snakemake allows one to generate a flow chart of the execution plan, that allow to visualize everything that has been done or needs to be done.

In [ ]:
!snakemake --dag | dot -Tsvg > dag.svg

In [8]:
from IPython.display import SVG
SVG('./dag.svg')

ExpatError: not well-formed (invalid token): line 1, column 1

## Provenance registry

Another very useful function if the **provenance** registry, that keeps track of which files have been created by which rule, when and with which parameters.

This allows to trace the origin of each file in an easy way.

It can also be set to append the provenance to the complete log, giving the complete history of all files and how they have been changed over time.

In [ ]:
!snakemake --detailed-summary > provenance.tsv

In [ ]:
import pandas as pd
pd.read_csv("provenance.tsv", index_col=0, sep='\t')

## Parallel execution

If I want to execute several rules at the same time (obviously maintaining the execution oerder needed for the files to be generated) I just need to add the --cores `<N>` and snakemake will execute automatically in parallel everything it can, given the available number of processors defined.

There is an equivalent way to launch the execution on a cluster job queue, making distributed computing really easy.

In [ ]:
!rm *.txt

In [ ]:
!snakemake --cores 6

I can also specify limited resources (alongside the cpu cores) so that the pipeline does not exceed these limits.

For example, if one has rules that require massive amount of memory, it is better to avoid launching them all the same time. I can specify an expected mount of needed memory and a total available one, and snakemake will make sure to not overschedule the execution to never surpass those limits.

In [ ]:
!rm *.txt

In [ ]:
%%file Snakefile

numbers = [1, 2, 3, 4]

rule all:
    input:
        expand("partial_{number}.txt", number=numbers)
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"
        
rule create_partials:
    output:
        out = "partial_{number}.txt"
    resources: 
        memory = 6
    run:
        filename = output.out
        with open(filename, 'w') as file:
            print("the result of {}".format(filename), file=file)

In [ ]:
!snakemake --cores 6 --resources memory=12

## Configurations

If there is the need to pass some configuration parameters, these can be given from the command line or loaded as a configuration file in YAML or JSON format.

Qui specifico con un range gli indici dei file.

In [ ]:
%%file Snakefile

numbers = [i for i in range(int(config['number']))]

rule all:
    input:
        expand("partial_{number}.txt", number=numbers)
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"
        
rule create_partials:
    output:
        out = "partial_{number}.txt"
    resources: 
        memory = 6
    run:
        filename = output.out
        with open(filename, 'w') as file:
            print("the result of {}".format(filename), file=file)

In [ ]:
!rm *.txt

In [ ]:
!snakemake --cores 6 --resources memory=12 --config number=4

In [ ]:
%%file config.yaml
number: 4

In [ ]:
%%file Snakefile

configfile: "./config.yaml"

numbers = [i for i in range(int(config['number']))]

rule all:
    input:
        expand("partial_{number}.txt", number=numbers)
    output:
        "result.txt"
    shell:
        "cat {input} > {output}"
        
rule create_partials:
    output:
        out = "partial_{number}.txt"
    resources: 
        memory = 6
    run:
        filename = output.out
        with open(filename, 'w') as file:
            print("the result of {}".format(filename), file=file)

In [ ]:
!rm *.txt

In [ ]:
!snakemake --cores 6 --resources memory=12

## Conda integration

In order to ensure reproducibility, it is also possible to define isolated software environments per rule.

Posso creare degli ambienti virtuali dedicati ad ogni rule.

In [ ]:
rule mytask:
    input:
        "path/to/{dataset}.txt"
    output:
        "result/{dataset}.txt"
    conda:
        "some-tool.yaml"
    shell:
        "some-tool {input} > {output}"

with the following environment definition:

In [ ]:
%%file some-tool.yaml
channels:
    - conda-forge
dependencies:
    - some-tool =2.3.1
    - some-lib =1.1.2

I just need to add the flag --use-conda to the workflow execution command, e.g. snakemake --cores 6 --use-conda and snakemake will automatically create the required software environments while executing the rule.

## Exercise

In the website (github) you can find some files that contain imaginary financial transactions of several people, stored as a .`tsv` files, where the first column contains the name of the person (unique to that person) and the second column store the amount that that person earned for that day.

There will also be a file storing the md5sum for each one of these files.

The exercise is in two parts:

1. Write a pipeline that download each of these files, check that the md5 hash is the expected one.
2. Load the data and merge them, summing the transactions for each person in a single value.

Notes:

the data can be found at the folder https://github.com/UniboDIFABiophysics/programmingCourseDIFA/tree/master/snakemake_exercise
with filenames `transazioni_{}.tsv` with an index from 00 to 49
the file with the hash of each file is in `md5sums.tsv`


### Suggestions

* The has function can be implemented in python or done using the bash `md5sum` program.
* The download can be done using the `requests` library or the `wget` program from terminal.
* Use the snakemake wildcards, or you will have to go crazy.

In [ ]:
import requests

url_base = ("https://github.com/UniboDIFABiophysics/programmingCourseDIFA/tree/master/snakemake_exercise/")
filename = "transazioni_00.tsv"

response = requests.get(url_base+filename)

# Throw an error for bad status codes
response.raise_for_status()

with open(filename, 'wb') as handle:
    handle.write(response.content)